# Autoencoders: Learning to Compress and Reconstruct

This notebook introduces **autoencoders**, a fundamental unsupervised learning architecture that learns to compress data into a compact representation and then reconstruct it. We'll build deep intuition for how autoencoders work, what they learn, and why they're useful across many domains.

## Learning Objectives

By the end of this notebook, you will:
- Understand the encoder-decoder architecture and the bottleneck principle
- Build and train a complete autoencoder from scratch
- Visualize and interpret learned latent representations
- Perform latent space interpolation to generate smooth transitions
- Apply autoencoders to practical tasks like denoising and compression
- Understand the foundations needed for Variational Autoencoders (VAEs)

## 1. Introduction: What is an Autoencoder?

Imagine you need to send images through a very narrow pipe. How do you compress them without losing important information?

An **autoencoder** learns to do exactly this:
1. **Compress** the input into a smaller representation (encoding)
2. **Reconstruct** the original input from this compressed form (decoding)

The magic: the network learns what information is most important to keep!

### The Core Idea

An autoencoder has three key components:

```
Input (784 dims)  →  [Encoder]  →  Latent Code (32 dims)  →  [Decoder]  →  Output (784 dims)
   [High-D]                            [Low-D]                                [High-D]
                                     ↑ BOTTLENECK ↑
```

**Key insight:** The bottleneck forces the network to learn a compressed representation that captures the essential features of the data.

**Training objective:** Minimize reconstruction error:
$$\mathcal{L} = \|x - \hat{x}\|^2$$

where $x$ is the input and $\hat{x}$ is the reconstruction.

### Why Are Autoencoders Useful?

- **Dimensionality Reduction**: Like PCA, but non-linear and more powerful
- **Feature Learning**: Discover useful representations without labels
- **Denoising**: Remove noise while preserving important features
- **Anomaly Detection**: Unusual inputs reconstruct poorly
- **Compression**: Reduce storage and transmission costs
- **Foundation for VAEs**: Understanding autoencoders is essential for generative models

## 2. Setup and Data Preparation

### Import Required Libraries

We'll use PyTorch for building and training our autoencoder, and work with the MNIST dataset of handwritten digits.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from tqdm.auto import tqdm

# Shared utilities
from aiml_notebooks import get_device, set_seed

%load_ext autoreload
%autoreload 2

print("✓ Libraries imported successfully")

### Set Random Seeds for Reproducibility

This ensures our results are consistent across runs.

In [ ]:
SEED = 42
set_seed(SEED)

print(f"Random seed set to {SEED}")

### Configure Device (GPU/CPU)

We'll automatically use GPU if available for faster training.

In [ ]:
device = get_device()
print(f"Using device: {device}")

### Load MNIST Dataset

MNIST contains 28×28 grayscale images of handwritten digits (0-9). This is perfect for learning about autoencoders because:
- Images are simple and interpretable
- 784 dimensions (28×28) compress nicely
- Fast training iterations

In [ ]:
# Transform images to tensors (values in [0, 1])
transform = transforms.Compose([
    transforms.ToTensor(),
])

# Download and load datasets
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Create data loaders
BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Training samples: {len(train_dataset):,}")
print(f"Test samples: {len(test_dataset):,}")
print(f"Image shape: {train_dataset[0][0].shape}")  # (1, 28, 28)
print(f"Batch size: {BATCH_SIZE}")

### Visualize Sample Images

Let's see what we're working with.

In [ ]:
# Display a grid of sample images
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f"Label: {label}", fontsize=9)
    ax.axis('off')
plt.suptitle('Sample MNIST Digits', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Each image is 28×28 pixels (784 total values)")
print("Our autoencoder will compress these 784 values into a much smaller representation!")

## 3. Building an Autoencoder from Scratch

### Understanding the Architecture

Our autoencoder will have a simple feedforward architecture:

**Encoder (Compression):**
```
Input (784) → Linear(512) → ReLU → Linear(256) → ReLU → Linear(32)
```

**Decoder (Reconstruction):**
```
Latent (32) → Linear(256) → ReLU → Linear(512) → ReLU → Linear(784) → Sigmoid
```

**Key design choices:**
- **Latent dimension (32)**: The bottleneck size - controls compression ratio
- **ReLU activations**: Non-linearity lets us learn complex patterns
- **Sigmoid output**: Ensures output values are in [0, 1] like our inputs
- **Symmetric architecture**: Decoder mirrors encoder (common but not required)

### Implement the Autoencoder

Let's build it step by step.

In [ ]:
class Autoencoder(nn.Module):
    """Simple feedforward autoencoder."""
    
    def __init__(self, input_dim=784, latent_dim=32):
        super(Autoencoder, self).__init__()
        
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        
        # Encoder: compresses input to latent representation
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim)
        )
        
        # Decoder: reconstructs input from latent representation
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, input_dim),
            nn.Sigmoid()  # Output in [0, 1]
        )
    
    def encode(self, x):
        """Encode input to latent representation."""
        # Flatten: (batch, 1, 28, 28) → (batch, 784)
        x = x.view(x.size(0), -1)
        return self.encoder(x)
    
    def decode(self, z):
        """Decode latent representation to reconstruction."""
        x = self.decoder(z)
        # Reshape: (batch, 784) → (batch, 1, 28, 28)
        return x.view(x.size(0), 1, 28, 28)
    
    def forward(self, x):
        """Full forward pass: encode then decode."""
        z = self.encode(x)
        x_recon = self.decode(z)
        return x_recon, z

print("✓ Autoencoder class defined")

### Create and Inspect the Model

Let's instantiate our autoencoder and see its structure.

In [ ]:
# Create model with latent dimension of 32
LATENT_DIM = 32
model = Autoencoder(input_dim=784, latent_dim=LATENT_DIM).to(device)

print(model)
print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Compression ratio: {784 / LATENT_DIM:.1f}× (from 784 to {LATENT_DIM} dimensions)")

### Test Forward Pass

Let's verify our model works by passing a single batch through it.

In [ ]:
# Get a batch of data
sample_batch, _ = next(iter(train_loader))
sample_batch = sample_batch.to(device)

# Forward pass
with torch.no_grad():
    reconstruction, latent = model(sample_batch)

print(f"Input shape:          {sample_batch.shape}")  # (batch_size, 1, 28, 28)
print(f"Latent shape:         {latent.shape}")         # (batch_size, 32)
print(f"Reconstruction shape: {reconstruction.shape}")  # (batch_size, 1, 28, 28)
print(f"\n✓ Forward pass successful!")

## 4. Training the Autoencoder

### Choosing a Loss Function

We need to measure how well we reconstruct the input. Two common choices:

**1. Mean Squared Error (MSE):**
$$\mathcal{L}_{MSE} = \frac{1}{n}\sum_{i=1}^{n}(x_i - \hat{x}_i)^2$$

**2. Binary Cross-Entropy (BCE):**
$$\mathcal{L}_{BCE} = -\sum_{i=1}^{n}[x_i \log(\hat{x}_i) + (1-x_i)\log(1-\hat{x}_i)]$$

For images normalized to [0, 1], **BCE often works better** because it treats each pixel as a probability. We'll use BCE in this notebook.

### Define Training Functions

We'll create functions to train for one epoch and evaluate on the test set.

In [ ]:
def train_epoch(model, train_loader, optimizer, device):
    """Train the autoencoder for one epoch."""
    model.train()
    total_loss = 0
    
    for batch_idx, (data, _) in enumerate(train_loader):
        data = data.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        reconstruction, _ = model(data)
        
        # Compute reconstruction loss
        loss = F.binary_cross_entropy(reconstruction, data, reduction='sum') / data.size(0)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)

def evaluate(model, test_loader, device):
    """Evaluate the autoencoder on test set."""
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for data, _ in test_loader:
            data = data.to(device)
            reconstruction, _ = model(data)
            loss = F.binary_cross_entropy(reconstruction, data, reduction='sum') / data.size(0)
            total_loss += loss.item()
    
    return total_loss / len(test_loader)

print("✓ Training functions defined")

### Train the Model

Now we train our autoencoder! This should take just a few minutes.

In [ ]:
# Training hyperparameters
LEARNING_RATE = 1e-3
NUM_EPOCHS = 20

# Create optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Track losses
train_losses = []
test_losses = []

# Training loop
print(f"Training for {NUM_EPOCHS} epochs...\n")
for epoch in tqdm(range(NUM_EPOCHS), desc="Training"):
    train_loss = train_epoch(model, train_loader, optimizer, device)
    test_loss = evaluate(model, test_loader, device)
    
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS} - Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}")

print("\n✓ Training complete!")

### Visualize Training Progress

Let's see how the loss decreased during training.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss', linewidth=2, marker='o', markersize=4)
plt.plot(test_losses, label='Test Loss', linewidth=2, marker='s', markersize=4)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss (BCE)', fontsize=12)
plt.title('Autoencoder Training Progress', fontsize=14)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final train loss: {train_losses[-1]:.4f}")
print(f"Final test loss:  {test_losses[-1]:.4f}")
print(f"\nLoss decreased by {(train_losses[0] - train_losses[-1]) / train_losses[0] * 100:.1f}% during training!")

## 5. Analyzing Reconstruction Quality

### Visualize Reconstructions

The moment of truth: how well does our autoencoder reconstruct images?

In [ ]:
def visualize_reconstructions(model, dataset, device, n_samples=10):
    """Display original and reconstructed images side by side."""
    model.eval()
    
    # Get random samples
    indices = np.random.choice(len(dataset), n_samples, replace=False)
    images = torch.stack([dataset[i][0] for i in indices]).to(device)
    labels = [dataset[i][1] for i in indices]
    
    # Reconstruct
    with torch.no_grad():
        reconstructions, _ = model(images)
    
    # Plot
    fig, axes = plt.subplots(2, n_samples, figsize=(n_samples*1.5, 3.5))
    
    for i in range(n_samples):
        # Original
        axes[0, i].imshow(images[i].cpu().squeeze(), cmap='gray')
        axes[0, i].set_title(f"Label: {labels[i]}", fontsize=9)
        axes[0, i].axis('off')
        if i == 0:
            axes[0, i].set_ylabel('Original', fontsize=12, rotation=0, ha='right', va='center')
        
        # Reconstruction
        axes[1, i].imshow(reconstructions[i].cpu().squeeze(), cmap='gray')
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].set_ylabel('Reconstructed', fontsize=12, rotation=0, ha='right', va='center')
    
    plt.suptitle('Original vs Reconstructed Images', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

visualize_reconstructions(model, test_dataset, device, n_samples=10)

### Observations

Notice that:
- The reconstructions capture the overall structure of the digits
- Some fine details may be lost or blurred
- The model learned to compress 784 dimensions into just 32!

This information bottleneck forces the autoencoder to learn the **most important features** for reconstruction.

### Compute Reconstruction Error

Let's quantify the reconstruction quality across the entire test set.

In [ ]:
def compute_reconstruction_errors(model, dataset, device, n_samples=1000):
    """Compute per-image reconstruction errors."""
    model.eval()
    errors = []
    
    # Sample subset for efficiency
    subset = Subset(dataset, range(min(n_samples, len(dataset))))
    loader = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=False)
    
    with torch.no_grad():
        for data, _ in loader:
            data = data.to(device)
            reconstruction, _ = model(data)
            
            # MSE per image
            batch_errors = F.mse_loss(reconstruction, data, reduction='none')
            batch_errors = batch_errors.view(data.size(0), -1).mean(dim=1)
            errors.extend(batch_errors.cpu().numpy())
    
    return np.array(errors)

# Compute errors
errors = compute_reconstruction_errors(model, test_dataset, device)

# Visualize distribution
plt.figure(figsize=(10, 5))
plt.hist(errors, bins=50, edgecolor='black', alpha=0.7)
plt.axvline(errors.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {errors.mean():.4f}')
plt.xlabel('Reconstruction Error (MSE)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Distribution of Reconstruction Errors', fontsize=14)
plt.legend(fontsize=11)
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"Mean error: {errors.mean():.6f}")
print(f"Std error:  {errors.std():.6f}")
print(f"Min error:  {errors.min():.6f}")
print(f"Max error:  {errors.max():.6f}")

## 6. Exploring the Latent Space

### What is the Latent Space?

The **latent space** (also called **latent representation** or **embedding space**) is the compressed representation learned by the encoder.

For our autoencoder:
- **Input**: 784 dimensions (28×28 pixels)
- **Latent**: 32 dimensions (compressed)
- **Output**: 784 dimensions (reconstructed)

The latent space captures the **essential features** needed to reconstruct the input. Let's explore what it learned!

### Extract Latent Representations

We'll encode a subset of test images into their latent representations.

In [ ]:
def extract_latents(model, dataset, device, max_samples=5000):
    """Extract latent vectors and labels for visualization."""
    model.eval()
    latents = []
    labels = []
    
    # Use subset for faster computation
    subset = Subset(dataset, range(min(max_samples, len(dataset))))
    loader = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=False)
    
    with torch.no_grad():
        for data, label in tqdm(loader, desc="Extracting latents"):
            data = data.to(device)
            z = model.encode(data)
            latents.append(z.cpu().numpy())
            labels.append(label.numpy())
    
    latents = np.concatenate(latents, axis=0)
    labels = np.concatenate(labels, axis=0)
    
    return latents, labels

# Extract latent representations
latents, labels = extract_latents(model, test_dataset, device)

print(f"Latent representations shape: {latents.shape}")  # (n_samples, 32)
print(f"Labels shape: {labels.shape}")                    # (n_samples,)

### Visualize Latent Space with t-SNE

Since we can't visualize 32 dimensions directly, we'll use **t-SNE** to reduce to 2D while preserving local structure.

In [ ]:
# Apply t-SNE to reduce from 32D to 2D
print("Running t-SNE (this may take a minute)...")
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30)
latents_2d = tsne.fit_transform(latents)

# Plot
plt.figure(figsize=(12, 10))
scatter = plt.scatter(latents_2d[:, 0], latents_2d[:, 1], 
                     c=labels, cmap='tab10', alpha=0.6, s=10)
plt.colorbar(scatter, label='Digit Class', ticks=range(10))
plt.title('Autoencoder Latent Space (t-SNE Visualization)', fontsize=14)
plt.xlabel('t-SNE Dimension 1', fontsize=12)
plt.ylabel('t-SNE Dimension 2', fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Each color represents a different digit (0-9)")
print("Notice how similar digits cluster together!")

### Key Observation: Clustering

**What we see:**
- Similar digits cluster together in latent space
- The autoencoder learned semantic structure without any labels!
- This is **unsupervised learning** at work

**Why this happens:**
- To reconstruct similar inputs well, the encoder maps them to similar latent codes
- The bottleneck forces the network to organize the latent space efficiently

### Visualize with PCA

Let's also try **PCA** (Principal Component Analysis), a linear dimensionality reduction method.

In [ ]:
# Apply PCA to reduce from 32D to 2D
pca = PCA(n_components=2)
latents_pca = pca.fit_transform(latents)

# Plot
plt.figure(figsize=(12, 10))
scatter = plt.scatter(latents_pca[:, 0], latents_pca[:, 1], 
                     c=labels, cmap='tab10', alpha=0.6, s=10)
plt.colorbar(scatter, label='Digit Class', ticks=range(10))
plt.title('Autoencoder Latent Space (PCA Visualization)', fontsize=14)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Total variance explained by first 2 PCs: {pca.explained_variance_ratio_.sum()*100:.1f}%")
print(f"\nPCA is linear, t-SNE is non-linear - both reveal different aspects of the structure")

## 7. Latent Space Interpolation

### Understanding Interpolation

Since the decoder maps latent vectors to images, we can:
1. Take two images: $x_1$ and $x_2$
2. Encode them: $z_1 = f_{enc}(x_1)$, $z_2 = f_{enc}(x_2)$
3. Create intermediate vectors: $z_t = (1-t) \cdot z_1 + t \cdot z_2$ for $t \in [0, 1]$
4. Decode each $z_t$ to get images: $\hat{x}_t = f_{dec}(z_t)$

This shows us **smooth transitions** between different images in latent space!

### Implement Interpolation

Let's create a function to interpolate between two images.

In [ ]:
def interpolate_images(model, img1, img2, device, n_steps=10):
    """Interpolate between two images in latent space."""
    model.eval()
    
    with torch.no_grad():
        # Encode both images
        img1 = img1.unsqueeze(0).to(device)
        img2 = img2.unsqueeze(0).to(device)
        z1 = model.encode(img1)
        z2 = model.encode(img2)
        
        # Create interpolation steps
        interpolations = []
        for t in np.linspace(0, 1, n_steps):
            z_t = (1 - t) * z1 + t * z2
            img_t = model.decode(z_t)
            interpolations.append(img_t.cpu().squeeze())
    
    return interpolations

print("✓ Interpolation function defined")

### Visualize Interpolation Between Different Digits

Let's see what happens when we smoothly move from one digit to another in latent space.

In [ ]:
# Find two different digits
idx1 = np.random.choice([i for i in range(len(test_dataset)) if test_dataset[i][1] == 3])
idx2 = np.random.choice([i for i in range(len(test_dataset)) if test_dataset[i][1] == 8])

img1, label1 = test_dataset[idx1]
img2, label2 = test_dataset[idx2]

# Interpolate
interpolations = interpolate_images(model, img1, img2, device, n_steps=10)

# Visualize
fig, axes = plt.subplots(1, len(interpolations), figsize=(15, 2))
for i, ax in enumerate(axes):
    ax.imshow(interpolations[i], cmap='gray')
    ax.axis('off')
    if i == 0:
        ax.set_title(f'{label1}', fontsize=12, fontweight='bold', color='blue')
    elif i == len(interpolations) - 1:
        ax.set_title(f'{label2}', fontsize=12, fontweight='bold', color='red')

plt.suptitle(f'Latent Space Interpolation: {label1} → {label2}', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

print(f"Smoothly morphing from '{label1}' to '{label2}' by interpolating latent codes")

### Interpolate Between Similar Digits

Let's also try interpolating between two instances of the same digit.

In [ ]:
# Find two different instances of the same digit
target_digit = 7
indices = [i for i in range(len(test_dataset)) if test_dataset[i][1] == target_digit]
idx1, idx2 = np.random.choice(indices, size=2, replace=False)

img1, label1 = test_dataset[idx1]
img2, label2 = test_dataset[idx2]

# Interpolate
interpolations = interpolate_images(model, img1, img2, device, n_steps=10)

# Visualize
fig, axes = plt.subplots(1, len(interpolations), figsize=(15, 2))
for i, ax in enumerate(axes):
    ax.imshow(interpolations[i], cmap='gray')
    ax.axis('off')
    if i == 0 or i == len(interpolations) - 1:
        ax.set_title(f'{label1}', fontsize=12, fontweight='bold')

plt.suptitle(f'Latent Space Interpolation: Two Different {target_digit}s', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

print(f"Interpolating between two different handwriting styles of the digit '{target_digit}'")

### Reflection on Interpolation

**What we observe:**
- Smooth transitions between digits
- Intermediate images may look blurry or like blends
- Some transitions look more realistic than others

**Why intermediate images can look unrealistic:**
- The latent space is not perfectly structured
- Some regions may not correspond to realistic digits
- This motivates **Variational Autoencoders (VAEs)** which explicitly structure the latent space

## 8. Application: Image Denoising

**Important:** Our current autoencoder was trained on clean images only. For effective denoising, we need to train a **denoising autoencoder** that learns from noisy inputs.

### How Denoising Autoencoders Work

A **denoising autoencoder** learns to remove noise by training on corrupted inputs:
1. **Training**: Input = noisy images, Target = clean images
2. The bottleneck forces the encoder to learn robust features that ignore noise
3. The decoder learns to reconstruct the clean version

**Key difference from regular autoencoders:**
- Regular: Trained on clean → clean (reconstruction)
- Denoising: Trained on noisy → clean (denoising)

Let's train a denoising autoencoder from scratch!

### Define Noise Function and Denoising Training

First, we need a function to add noise, and a training function that uses noisy inputs with clean targets.

In [ ]:
def add_gaussian_noise(images, noise_factor=0.5):
    """Add Gaussian noise to images."""
    noisy = images + noise_factor * torch.randn_like(images)
    return torch.clamp(noisy, 0.0, 1.0)  # Keep values in [0, 1]

def train_denoising_epoch(model, train_loader, optimizer, device, noise_factor=0.5):
    """Train denoising autoencoder for one epoch."""
    model.train()
    total_loss = 0
    
    for batch_idx, (clean_data, _) in enumerate(train_loader):
        clean_data = clean_data.to(device)
        
        # Add noise to create corrupted input
        noisy_data = add_gaussian_noise(clean_data, noise_factor)
        
        # Forward pass: encode noisy, decode to clean
        optimizer.zero_grad()
        reconstruction, _ = model(noisy_data)
        
        # Compute loss: compare reconstruction to CLEAN target
        loss = F.binary_cross_entropy(reconstruction, clean_data, reduction='sum') / clean_data.size(0)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)

print("✓ Noise and denoising training functions defined")

### Train Denoising Autoencoder

Now let's train a new autoencoder specifically for denoising. We'll train it with noisy inputs and clean targets.

In [ ]:
# Create a new denoising autoencoder
denoising_model = Autoencoder(input_dim=784, latent_dim=LATENT_DIM).to(device)
denoising_optimizer = optim.Adam(denoising_model.parameters(), lr=LEARNING_RATE)

# Training parameters
NOISE_FACTOR = 0.5
DENOISING_EPOCHS = 10

print(f"Training denoising autoencoder for {DENOISING_EPOCHS} epochs with noise factor {NOISE_FACTOR}...\n")

denoising_losses = []
for epoch in tqdm(range(DENOISING_EPOCHS), desc="Training denoising AE"):
    loss = train_denoising_epoch(denoising_model, train_loader, denoising_optimizer, device, NOISE_FACTOR)
    denoising_losses.append(loss)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/{DENOISING_EPOCHS} - Loss: {loss:.4f}")

print("\n✓ Denoising autoencoder training complete!")

### Visualize Denoising Results

Now let's test our denoising autoencoder on new noisy images.

### Why Denoising Works

The denoising autoencoder learned to:
1. **Recognize patterns**: The encoder identifies digit structure despite noise
2. **Filter noise**: The bottleneck compresses only essential features, discarding random noise
3. **Reconstruct clean**: The decoder generates clean images from the noise-free latent representation

**Key insight:** By training on noisy → clean pairs, the network learns which features are signal (digit structure) and which are noise (random variations).

In [ ]:
# Get test samples and add noise
n_samples = 8
indices = np.random.choice(len(test_dataset), n_samples, replace=False)
clean_images = torch.stack([test_dataset[i][0] for i in indices]).to(device)
labels = [test_dataset[i][1] for i in indices]

# Add noise
noisy_images = add_gaussian_noise(clean_images, noise_factor=NOISE_FACTOR)

# Denoise using our trained denoising autoencoder
denoising_model.eval()
with torch.no_grad():
    denoised_images, _ = denoising_model(noisy_images)

# Visualize: Clean, Noisy, Denoised
fig, axes = plt.subplots(3, n_samples, figsize=(n_samples*1.5, 5))

for i in range(n_samples):
    # Clean
    axes[0, i].imshow(clean_images[i].cpu().squeeze(), cmap='gray')
    axes[0, i].set_title(f"Label: {labels[i]}", fontsize=9)
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('Clean', fontsize=12, rotation=0, ha='right', va='center')
    
    # Noisy
    axes[1, i].imshow(noisy_images[i].cpu().squeeze(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('Noisy', fontsize=12, rotation=0, ha='right', va='center')
    
    # Denoised
    axes[2, i].imshow(denoised_images[i].cpu().squeeze(), cmap='gray')
    axes[2, i].axis('off')
    if i == 0:
        axes[2, i].set_ylabel('Denoised', fontsize=12, rotation=0, ha='right', va='center')

plt.suptitle('Denoising Autoencoder: Removing Noise from Images', fontsize=14, y=0.98)
plt.tight_layout()
plt.show()

print(f"Successfully denoised images trained with noise factor {NOISE_FACTOR}!")

## 9. Application: Anomaly Detection

### The Anomaly Detection Principle

Autoencoders can detect anomalies based on this insight:
- **Normal data**: Reconstructs well (low error)
- **Anomalous data**: Reconstructs poorly (high error)

This works because the autoencoder learned to represent normal patterns. Anything unusual will have higher reconstruction error!

### Simulate Anomaly Detection

Let's pretend digit '9' is an anomaly and see if our autoencoder (trained on all digits) gives it higher reconstruction error.

In [ ]:
# Get normal samples (digits 0-8) and anomaly samples (digit 9)
normal_indices = [i for i in range(len(test_dataset)) if test_dataset[i][1] != 9]
anomaly_indices = [i for i in range(len(test_dataset)) if test_dataset[i][1] == 9]

# Sample from each
n_samples = 100
normal_samples = torch.stack([test_dataset[i][0] for i in np.random.choice(normal_indices, n_samples)]).to(device)
anomaly_samples = torch.stack([test_dataset[i][0] for i in np.random.choice(anomaly_indices, n_samples)]).to(device)

# Compute reconstruction errors
model.eval()
with torch.no_grad():
    recon_normal, _ = model(normal_samples)
    recon_anomaly, _ = model(anomaly_samples)
    
    # MSE per image
    normal_errors = F.mse_loss(recon_normal, normal_samples, reduction='none')
    normal_errors = normal_errors.view(n_samples, -1).mean(dim=1).cpu().numpy()
    
    anomaly_errors = F.mse_loss(recon_anomaly, anomaly_samples, reduction='none')
    anomaly_errors = anomaly_errors.view(n_samples, -1).mean(dim=1).cpu().numpy()

print(f"Normal samples (0-8) - Mean error: {normal_errors.mean():.6f}")
print(f"Anomaly samples (9)  - Mean error: {anomaly_errors.mean():.6f}")
print(f"Ratio: {anomaly_errors.mean() / normal_errors.mean():.2f}×")

### Visualize Error Distributions

Let's plot the reconstruction error distributions for normal vs anomaly samples.

In [ ]:
plt.figure(figsize=(12, 5))

plt.hist(normal_errors, bins=30, alpha=0.7, label='Normal (0-8)', color='green', edgecolor='black')
plt.hist(anomaly_errors, bins=30, alpha=0.7, label='Anomaly (9)', color='red', edgecolor='black')

plt.axvline(normal_errors.mean(), color='darkgreen', linestyle='--', linewidth=2, 
            label=f'Normal Mean: {normal_errors.mean():.6f}')
plt.axvline(anomaly_errors.mean(), color='darkred', linestyle='--', linewidth=2,
            label=f'Anomaly Mean: {anomaly_errors.mean():.6f}')

plt.xlabel('Reconstruction Error (MSE)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Anomaly Detection: Normal vs Anomalous Reconstruction Errors', fontsize=14)
plt.legend(fontsize=10)
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\nNote: In this example, digit 9 doesn't show much higher error because the model")
print("was trained on all digits. For real anomaly detection, train ONLY on normal data!")

## 10. Experiments and Exploration

### Experiment: Effect of Latent Dimension

How does the bottleneck size affect reconstruction quality and compression?

**Try this:**
```python
# Extreme compression (2D latent space)
tiny_model = Autoencoder(input_dim=784, latent_dim=2).to(device)
# Train and visualize...
# With 2D latent, you can plot it directly without t-SNE!

# Larger capacity
large_model = Autoencoder(input_dim=784, latent_dim=128).to(device)
# Compare reconstruction quality
```

**Questions to explore:**
- What's the smallest latent dimension that still gives good reconstructions?
- Does reconstruction quality plateau at some point?
- How does latent dimension affect clustering in visualization?

### Experiment: Network Depth

Does making the encoder/decoder deeper improve performance?

**Try this:**
```python
# Deeper network
self.encoder = nn.Sequential(
    nn.Linear(784, 512),
    nn.ReLU(),
    nn.Linear(512, 384),
    nn.ReLU(),
    nn.Linear(384, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 32)
)
```

**Compare:**
- Training time
- Final reconstruction error
- Number of parameters

### Experiment: Convolutional Autoencoder

For images, convolutional layers work better than fully-connected layers.

**Try implementing:**
```python
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder: Conv2d layers that downsample
        # Decoder: ConvTranspose2d layers that upsample
```

**Compare:**
- Reconstruction quality
- Number of parameters
- Visual sharpness

## 11. Key Takeaways

### Core Concepts

1. **Autoencoders** learn to compress and reconstruct data through an encoder-decoder architecture

2. **The bottleneck** (latent space) forces the network to learn essential features

3. **Training objective**: Minimize reconstruction error (MSE or BCE)

4. **Latent space** captures semantic structure - similar inputs cluster together

5. **Interpolation** in latent space creates smooth transitions between inputs

6. **Applications**: Dimensionality reduction, denoising, anomaly detection, compression

7. **Unsupervised learning**: No labels needed - learns from data structure alone

### Limitations and Next Steps

**Limitations of basic autoencoders:**
- Latent space can have "holes" (regions that don't decode well)
- Interpolations may produce unrealistic outputs
- Cannot easily generate new samples
- No probabilistic interpretation

**Next steps:**
- **Variational Autoencoders (VAEs)**: Add probabilistic structure to latent space for better generation
- **Denoising Autoencoders**: Train explicitly with noisy inputs
- **Convolutional Autoencoders**: Better for image data
- **Vector-Quantized VAE (VQ-VAE)**: Discrete latent space
- **Adversarial Autoencoders**: Combine with GANs

### Connection to Other Concepts

**Autoencoders are foundational for:**
- **VAEs**: Probabilistic generative models (next in learning path!)
- **GANs**: Alternative generative approach
- **Representation learning**: Learning useful features
- **Dimensionality reduction**: Non-linear alternative to PCA
- **Self-supervised learning**: Pretraining representations
- **Anomaly detection**: Identifying unusual patterns

## Summary

Congratulations! You've built a complete understanding of autoencoders:

✅ Understood the encoder-decoder architecture and bottleneck principle  
✅ Implemented and trained an autoencoder from scratch  
✅ Visualized and analyzed the learned latent space  
✅ Performed smooth interpolations between images  
✅ Applied autoencoders to denoising and anomaly detection  
✅ Built foundations for understanding VAEs and generative models  

You now understand one of the most fundamental unsupervised learning architectures in deep learning. This knowledge is essential for advanced topics like Variational Autoencoders (VAEs), which we'll explore next in the learning path!